# Этап 3. Извлечение признаков

Вход: `data/tracks.csv`, `audio/*.wav`, `data/feature_schema.json` (этапы 1–2)

Выход: `data/features.csv`, `data/clap_embeddings.npy`

In [ ]:
%pip install -q "yt-dlp[default]" imageio-ffmpeg rapidfuzz librosa soundfile umap-learn transformers tqdm
%pip install -q deno

In [ ]:
import os, sys, json, time, re, subprocess, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)

IN_COLAB = 'google.colab' in sys.modules
USE_DRIVE = True

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/music_dataset')
else:
    BASE_DIR = Path('music_dataset').resolve()

AUDIO_DIR, DATA_DIR, FIG_DIR = BASE_DIR / 'audio', BASE_DIR / 'data', BASE_DIR / 'figures'
for d in (AUDIO_DIR, DATA_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

GENRE_ORDER = ['rock', 'pop', 'hip_hop', 'electronic', 'jazz', 'classical',
               'metal', 'rnb_soul', 'country', 'reggae', 'ambient', 'latin']
GENRE_RU = {'rock': 'рок', 'pop': 'поп', 'hip_hop': 'хип-хоп', 'electronic': 'электроника',
            'jazz': 'джаз', 'classical': 'классика', 'metal': 'метал', 'rnb_soul': 'R&B / соул',
            'country': 'кантри', 'reggae': 'регги', 'ambient': 'эмбиент', 'latin': 'латино'}

## 3. Пайплайн извлечения признаков

Схема такая: из каждого 30-секундного фрагмента считаю сигнальные признаки через librosa. Параллельно режу фрагмент на 3 окна по 10 секунд, прогоняю их через аудио-энкодер CLAP и усредняю, это и есть эмбеддинг трека. Потом сравниваю эмбеддинг с текстовыми эмбеддингами классов («energetic music», «music for a workout» и т. д.) и через softmax получаю вероятности настроения, контекста и атмосферы. Для инструментов и вокала сравниваю две фразы, «с X» и «без X».

**Что использую:**
- **CLAP** (`laion/larger_clap_music` с HuggingFace). Модель от LAION, обучена сопоставлять аудио и текст, эта версия дообучена на музыке. Из неё беру и эмбеддинги, и теги.
- **librosa** для темпа, громкости, спектральных признаков, разделения на гармоническую и ударную часть и хромаграммы. Тональность определяю по профилям Крумхансла.
- По желанию **LLM** (Anthropic API) для списка похожих артистов. По умолчанию выключено, нужен ключ.

**Почему CLAP.** Размеченных данных нет, а обучать классификатор настроения на 96 треках бессмысленно. CLAP позволяет задать классы просто текстом. Кроме того, теги считаются по самому звуку, а не по названию, поэтому подход сработает и для неизвестного трека. Окна по 10 секунд нужны, потому что модель обучалась на таких фрагментах.

Ещё одна деталь: CLAP склонен отдавать предпочтение некоторым классам почти для всех треков. Поэтому перед softmax я вычитаю из сходств среднее по датасету для каждого класса. Метки без этой поправки тоже сохраняю (`*_top_raw`), чтобы сравнить.

**Какие альтернативы рассматривала:**
- Essentia с моделями MTG (жанр, настроение, danceability). Классификаторы хорошие, но таксономии фиксированные, контекстов нет, и установка сложнее.
- MERT. Сильные эмбеддинги, но без текстовой части, теги пришлось бы учить на разметке.
- PANNs / AST (AudioSet). Хорошо находят инструменты, но с настроением справляются слабо.
- Только LLM по названию трека. Просто, но модель не слушает музыку, может выдумывать и ничего не знает о новых треках.
- Только librosa. Прозрачно, но семантику не ловит. Этот вариант я всё равно сравниваю на этапе 5.

In [ ]:
import librosa
from tqdm.auto import tqdm

tracks = pd.read_csv(DATA_DIR / 'tracks.csv')
schema = json.loads((DATA_DIR / 'feature_schema.json').read_text(encoding='utf-8'))
FEATURE_SCHEMA = schema['semantic']
tracks = tracks[tracks['track_id'].apply(lambda t: (AUDIO_DIR / f'{t}.wav').exists())].reset_index(drop=True)
print('Треков с аудио:', len(tracks))

SR_CLAP, SR_SIG = 48000, 22050
WIN_SEC, N_WIN = 10, 3

def load_audio(track_id):
    y, _ = librosa.load(str(AUDIO_DIR / f'{track_id}.wav'), sr=SR_CLAP, mono=True)
    return y.astype(np.float32)

def make_windows(y, sr=SR_CLAP, win_sec=WIN_SEC, n=N_WIN):
    L = int(win_sec * sr)
    if len(y) < L * n:
        y = np.pad(y, (0, L * n - len(y)))
    starts = np.linspace(0, len(y) - L, n).astype(int)
    return [y[s:s + L] for s in starts]

### 3.1 Признаки librosa

In [ ]:
MAJOR_PROFILE = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
MINOR_PROFILE = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17])
PITCHES = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']

def estimate_key(chroma_mean):
    best = (-2, None, None)
    for i in range(12):
        for prof, mode in ((MAJOR_PROFILE, 1), (MINOR_PROFILE, 0)):
            r = np.corrcoef(chroma_mean, np.roll(prof, i))[0, 1]
            if r > best[0]:
                best = (r, i, mode)
    r, i, mode = best
    return f"{PITCHES[i]} {'major' if mode else 'minor'}", mode, float(r)

def signal_features(y48):
    y = librosa.resample(y48, orig_sr=SR_CLAP, target_sr=SR_SIG)
    sr, hop = SR_SIG, 512
    dur = len(y) / sr
    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop)
    tempo, _ = librosa.beat.beat_track(onset_envelope=onset_env, sr=sr, hop_length=hop)
    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    onsets = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sr, hop_length=hop)
    # ритмичность: пик автокорреляции огибающей атак в диапазоне 40-200 BPM
    oe = onset_env - onset_env.mean()
    ac = librosa.autocorrelate(oe)
    ac = ac / (ac[0] + 1e-9)
    lag_min, lag_max = int(60 / 200 * sr / hop), int(60 / 40 * sr / hop)
    pulse = float(np.max(ac[lag_min:lag_max])) if len(ac) > lag_max else np.nan
    y_harm, y_perc = librosa.effects.hpss(y)
    chroma = librosa.feature.chroma_cqt(y=y_harm, sr=sr, hop_length=hop).mean(axis=1)
    key, mode, key_strength = estimate_key(chroma)
    p95, p5 = np.percentile(rms, 95), np.percentile(rms, 5)
    return {
        'tempo_bpm': float(np.atleast_1d(tempo)[0]),
        'rms_mean': float(rms.mean()),
        'rms_dynamic_range_db': float(20 * np.log10((p95 + 1e-6) / (p5 + 1e-6))),
        'spectral_centroid_hz': float(librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0].mean()),
        'spectral_flatness': float(librosa.feature.spectral_flatness(y=y, hop_length=hop)[0].mean()),
        'onset_rate': len(onsets) / dur,
        'pulse_clarity': pulse,
        'percussive_ratio': float(np.sum(y_perc ** 2) / (np.sum(y_harm ** 2) + np.sum(y_perc ** 2) + 1e-9)),
        'zcr_mean': float(librosa.feature.zero_crossing_rate(y, hop_length=hop)[0].mean()),
        'key': key, 'mode_major': mode, 'key_strength': key_strength,
    }

audio_cache, sig_rows = {}, []
for t in tqdm(tracks['track_id']):
    y = load_audio(t)
    audio_cache[t] = y
    sig_rows.append({'track_id': t, **signal_features(y)})
signal_df = pd.DataFrame(sig_rows)
signal_df.describe().T.round(3)

### 3.2 CLAP
Лучше запускать с GPU (Среда выполнения → Сменить среду → T4 GPU), на CPU тоже работает, но медленнее.

In [ ]:
import torch
from transformers import ClapModel, ClapProcessor

CLAP_ID = 'laion/larger_clap_music'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
clap_model = ClapModel.from_pretrained(CLAP_ID).to(device).eval()
clap_proc = ClapProcessor.from_pretrained(CLAP_ID)

# в новых версиях transformers get_*_features возвращает не тензор, а объект
def _embeds(out, kind):
    if torch.is_tensor(out):
        return out
    for attr in (f'{kind}_embeds', 'pooler_output'):
        v = getattr(out, attr, None)
        if v is not None:
            return v
    raise TypeError(type(out))

@torch.no_grad()
def clap_embed_audio(waves, batch_size=8):
    out = []
    for i in range(0, len(waves), batch_size):
        inp = clap_proc.feature_extractor(waves[i:i + batch_size], sampling_rate=SR_CLAP, return_tensors='pt')
        inp = {k: v.to(device) for k, v in inp.items()}
        e = _embeds(clap_model.get_audio_features(**inp), 'audio')
        out.append(torch.nn.functional.normalize(e, dim=-1).cpu().numpy())
    return np.concatenate(out)

@torch.no_grad()
def clap_embed_text(texts):
    inp = clap_proc.tokenizer(texts, padding=True, return_tensors='pt')
    inp = {k: v.to(device) for k, v in inp.items() if k in ('input_ids', 'attention_mask')}
    e = _embeds(clap_model.get_text_features(**inp), 'text')
    return torch.nn.functional.normalize(e, dim=-1).cpu().numpy()

try:
    LOGIT_SCALE = float(clap_model.logit_scale_a.exp().item())
except Exception:
    LOGIT_SCALE = 30.0
print(device, LOGIT_SCALE)

In [ ]:
win_list, owner = [], []
for i, t in enumerate(tracks['track_id']):
    for w in make_windows(audio_cache[t]):
        win_list.append(w); owner.append(i)
win_emb = clap_embed_audio(win_list)
owner = np.array(owner)
E = np.stack([win_emb[owner == i].mean(0) for i in range(len(tracks))])
E = E / np.linalg.norm(E, axis=1, keepdims=True)
win_consistency = np.array([(win_emb[owner == i] @ E[i]).mean() for i in range(len(tracks))])
print(E.shape, 'средняя близость окон к треку:', win_consistency.mean().round(3))

### 3.3 Zero-shot теги

In [ ]:
def softmax(z, axis=1):
    z = z - z.max(axis=axis, keepdims=True)
    ez = np.exp(z)
    return ez / ez.sum(axis=axis, keepdims=True)

def text_prototypes(spec):
    protos = []
    for lab, txt in spec['labels'].items():
        T = clap_embed_text([tpl.format(txt) for tpl in spec['templates']])
        p = T.mean(0)
        protos.append(p / np.linalg.norm(p))
    return np.stack(protos)

def zero_shot_multiclass(E, spec, debias=True):
    sims = E @ text_prototypes(spec).T
    raw = softmax(LOGIT_SCALE * sims)
    if debias:
        sims = sims - sims.mean(0, keepdims=True)
    return softmax(LOGIT_SCALE * sims), raw

def zero_shot_binary(E, spec):
    out = {}
    for lab, (pos, neg) in spec['labels'].items():
        T = clap_embed_text([pos, neg])
        out[lab] = softmax(LOGIT_SCALE * (E @ T.T))[:, 0]
    return out

sem = pd.DataFrame({'track_id': tracks['track_id']})
prefix = {'genre_clap': 'clapgenre', 'mood': 'mood', 'context': 'ctx', 'atmosphere': 'atm'}
for cat, pfx in prefix.items():
    spec = FEATURE_SCHEMA[cat]
    labels = list(spec['labels'])
    P, P_raw = zero_shot_multiclass(E, spec)
    for j, lab in enumerate(labels):
        sem[f'{pfx}_{lab}'] = P[:, j]
    order = np.argsort(-P, axis=1)
    sem[f'{pfx}_top'] = [labels[k] for k in order[:, 0]]
    sem[f'{pfx}_top2'] = [labels[k] for k in order[:, 1]]
    sem[f'{pfx}_top_conf'] = P.max(1)
    sem[f'{pfx}_top_raw'] = [labels[k] for k in P_raw.argmax(1)]

for lab, p in zero_shot_binary(E, FEATURE_SCHEMA['instrument']).items():
    sem[f'inst_{lab}'] = p
sem['vocals_prob'] = zero_shot_binary(E, FEATURE_SCHEMA['vocals'])['vocals']
sem['has_vocals'] = sem['vocals_prob'] > 0.5
sem[['track_id', 'mood_top', 'ctx_top', 'ctx_top2', 'atm_top', 'clapgenre_top', 'vocals_prob']].head(12)

### 3.4 Похожие артисты через LLM (необязательно)
Здесь LLM отвечает по своим знаниям о треке, а не по звуку. Включается флагом `USE_LLM`, ключ в Colab лежит в Secrets под именем `ANTHROPIC_API_KEY`.

In [ ]:
USE_LLM = False
LLM_MODEL = 'claude-haiku-5-5'
llm_path = DATA_DIR / 'llm_tags.json'

if USE_LLM:
    %pip install -q anthropic
    import anthropic
    if IN_COLAB:
        from google.colab import userdata
        os.environ.setdefault('ANTHROPIC_API_KEY', userdata.get('ANTHROPIC_API_KEY'))
    client = anthropic.Anthropic()
    llm = json.loads(llm_path.read_text(encoding='utf-8')) if llm_path.exists() else {}
    for row in tqdm(list(tracks.itertuples(index=False))):
        if row.track_id in llm:
            continue
        prompt = (f'Track: "{row.title}" by {row.artist} ({row.year}). Reply with JSON only: '
                  '{"similar_artists": [3 artists], "subgenre": "...", "mood_tags": [3 words]}')
        msg = client.messages.create(model=LLM_MODEL, max_tokens=200, messages=[{'role': 'user', 'content': prompt}])
        txt = msg.content[0].text
        try:
            llm[row.track_id] = json.loads(txt[txt.find('{'): txt.rfind('}') + 1])
        except Exception:
            llm[row.track_id] = {}
    llm_path.write_text(json.dumps(llm, ensure_ascii=False, indent=1), encoding='utf-8')

if llm_path.exists():
    llm = json.loads(llm_path.read_text(encoding='utf-8'))
    sem['llm_similar_artists'] = sem['track_id'].map(lambda t: '|'.join(llm.get(t, {}).get('similar_artists', [])))
    sem['llm_subgenre'] = sem['track_id'].map(lambda t: llm.get(t, {}).get('subgenre'))

### 3.5 Сохранение и быстрая проверка
Проверяю две вещи: насколько хорошо CLAP угадывает жанр (случайное угадывание даёт 1/12) и согласуются ли теги с сигнальными признаками. Например, у «энергичных» треков должна быть выше громкость.

In [ ]:
features = tracks.merge(signal_df, on='track_id').merge(sem, on='track_id')
features['clap_window_consistency'] = win_consistency
features.to_csv(DATA_DIR / 'features.csv', index=False)
np.save(DATA_DIR / 'clap_embeddings.npy', E)
pd.Series(tracks['track_id']).to_csv(DATA_DIR / 'clap_embeddings_ids.csv', index=False)
print(features.shape, E.shape)

acc = (features['clapgenre_top'] == features['genre']).mean()
acc_raw = (features['clapgenre_top_raw'] == features['genre']).mean()
top2 = ((features['clapgenre_top'] == features['genre']) | (features['clapgenre_top2'] == features['genre'])).mean()
print(f'жанр zero-shot: acc = {acc:.2f}, без поправки {acc_raw:.2f}, top-2 = {top2:.2f}')

from scipy.stats import spearmanr
checks = [('mood_energetic', 'rms_mean'), ('mood_calm', 'rms_mean'), ('ctx_workout', 'tempo_bpm'),
          ('ctx_sleep', 'onset_rate'), ('inst_drums', 'percussive_ratio'), ('mood_happy', 'mode_major')]
pd.DataFrame([{'тег': a, 'сигнал': b, 'spearman': round(spearmanr(features[a], features[b], nan_policy='omit')[0], 3)}
              for a, b in checks])